# 🛠️ Pré-processamento e Feature Engineering - Predição de Limite de Crédito

Este notebook tem como objetivo preparar os dados para modelagem, incluindo:

- Limpeza da base
- Tratamento de variáveis categóricas
- Escalonamento de variáveis numéricas
- Feature engineering
- Construção de datasets para modelagem

## 🎯 Target: Limit

In [3]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

import statsmodels.api as sm

In [9]:
df = pd.read_csv("../data/raw/Credit.csv")

df.head()

,index,Income,Limit,Rating,Cards,Age,Education,Gender,Student,Married,Ethnicity,Balance
0,1,14.891,3606,283,2,34,11,Male,No,Yes,Caucasian,333
1,2,106.025,6645,483,3,82,15,Female,Yes,Yes,Asian,903
2,3,104.593,7075,514,4,71,11,Male,No,No,Asian,580
3,4,148.924,9504,681,3,36,11,Female,No,No,Asian,964
4,5,55.882,4897,357,2,68,16,Male,No,Yes,Caucasian,331


In [10]:
df = df.drop(columns=["index"])  # coluna vazia

In [11]:
df.duplicated().sum()

np.int64(0)

In [12]:
df = df.drop_duplicates()

In [13]:
target = "Limit"

X = df.drop(columns=[target])
y = df[target]

In [14]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=42
)

In [15]:
categorical_cols = X_train.select_dtypes(include=["object"]).columns.tolist()
numerical_cols = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()

print("Categóricas:", categorical_cols)
print("Numéricas:", numerical_cols)

Categóricas: ['Gender', 'Student', 'Married', 'Ethnicity']
Numéricas: ['Income', 'Rating', 'Cards', 'Age', 'Education', 'Balance']


C:\Users\guilh\AppData\Local\Temp\ipykernel_5968\67137367.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X_train.select_dtypes(include=["object"]).columns.tolist()


In [16]:
if "Balance" in X_train.columns and "Income" in X_train.columns:
    X_train["debt_to_income"] = X_train["Balance"] / (X_train["Income"] + 1)
    X_test["debt_to_income"] = X_test["Balance"] / (X_test["Income"] + 1)

In [17]:
if "Cards" in X_train.columns and "Age" in X_train.columns:
    X_train["cards_per_age"] = X_train["Cards"] / (X_train["Age"] + 1)
    X_test["cards_per_age"] = X_test["Cards"] / (X_test["Age"] + 1)

In [18]:
y_train_log = np.log1p(y_train)
y_test_log = np.log1p(y_test)

In [19]:
numeric_transformer = Pipeline(steps=[
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("onehot", OneHotEncoder(drop="first", handle_unknown="ignore"))
])

In [20]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numerical_cols),
        ("cat", categorical_transformer, categorical_cols)
    ]
)

In [21]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

In [22]:
print("X_train:", X_train_processed.shape)
print("X_test:", X_test_processed.shape)

X_train: (300, 11)
X_test: (100, 11)
